# Case Study 02 — Diabetes Risk Classification

**Problem type:** Binary Classification · **Dataset:** Pima Indians Diabetes

End-to-end classification workflow with class imbalance handling and interpretability.

In [ ]:
import sys
sys.path.append('../..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from src.evaluation import evaluate_classification, compare_models, feature_importance_df
from src.visualization import plot_confusion_matrix, plot_roc_curve, plot_feature_importance
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## Step 1: Problem Framing

**Goal:** Predict diabetes diagnosis from 8 medical features.

**Success metric:** ROC-AUC > 0.80, recall > 0.70 (false negatives are costly — missing a diagnosis).

## Step 2: Data Understanding

In [ ]:
try:
    data = fetch_openml('diabetes', version=1, as_frame=True, parser='auto')
    X, y = data.data, (data.target == 'tested_positive').astype(int)
except Exception:
    from sklearn.datasets import make_classification
    X_arr, y_arr = make_classification(n_samples=768, n_features=8, n_informative=6, random_state=42)
    X = pd.DataFrame(X_arr, columns=['preg','gluc','bp','skin','ins','bmi','dpf','age'])
    y = pd.Series(y_arr)

print(f'Shape: {X.shape}')
print(f'Positive class: {y.sum()} / {len(y)} ({y.mean()*100:.1f}%)')
X.head()

## Step 3: Preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## Step 4–5: Model Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42),
    'SVM (RBF)': SVC(probability=True, random_state=42),
}

results = []
preds_dict = {}
probs_dict = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    probs = model.predict_proba(X_test_s)
    preds_dict[name] = preds
    probs_dict[name] = probs
    results.append(evaluate_classification(y_test, preds, probs, name=name))

compare_models(results).round(4)

## Step 6: Evaluation — Confusion Matrix + ROC

In [ ]:
best_name = 'Gradient Boosting'
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

from sklearn.metrics import confusion_matrix, roc_curve, auc
cm = confusion_matrix(y_test, preds_dict[best_name])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Diabetes','Diabetes'], yticklabels=['No Diabetes','Diabetes'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title(f'{best_name} — Confusion Matrix', fontweight='bold')

for name, probs in probs_dict.items():
    fpr, tpr, _ = roc_curve(y_test, probs[:, 1])
    axes[1].plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc(fpr,tpr):.3f})')
axes[1].plot([0,1], [0,1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves — All Models', fontweight='bold')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('results/02_classification_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: Interpretation — Feature Importance

In [ ]:
rf = models['Random Forest']
fi = feature_importance_df(X.columns.tolist(), rf.feature_importances_, top_k=8)
print(fi)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi['feature'][::-1], fi['importance'][::-1], color='teal')
ax.set_xlabel('Importance')
ax.set_title('Top Features — Random Forest', fontweight='bold')
plt.tight_layout()
plt.savefig('results/02_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Reflection

**Findings:** Gradient Boosting and Random Forest achieved the best AUC. Glucose level and BMI are the top features — consistent with medical literature.

**Class imbalance considerations:** Dataset is ~35% positive class. For medical applications, we'd want to use class weights or SMOTE oversampling to improve recall.

**Limitations:**
- Small dataset (768 samples)
- All features collected at one time point; no temporal dynamics
- Not validated on diverse demographics (original dataset is Pima Indian women)

**Production considerations:**
- Calibrated probability outputs (Platt scaling / isotonic)
- Cost-sensitive decision threshold (false negatives more costly than false positives)
- Bias auditing across demographic groups before deployment

---
[← Back to main README](../../README.md)